# ESP32-S3 Edge Acoustic Event Classifier: Knowledge Distillation Pipeline
This notebook executes the knowledge distillation workflow from YAMNet into an edge DS-CNN:
1. Ingest pretrained YAMNet teacher model weights and map AudioSet ontology classes to target categories (Dog, DJ, Traffic, Background).
2. Download and partition environmental acoustic data from the ESC-50 dataset.
3. Train the compact 4-class Depthwise-Separable CNN student model (~75 KB INT8, <96 KB SRAM footprint).
4. Perform Full-Integer INT8 Post-Training Quantization (PTQ) and export C source arrays to esp32s3_firmware/main/model_data.cc.

In [ ]:
!pip install -q tensorflow tensorflow-hub scipy soundfile
import os, urllib.request, zipfile
# Optional: Download ESC-50 environmental audio dataset for real-world Dog & Traffic recordings
if not os.path.exists('ESC-50-master'):
    print('Downloading ESC-50 dataset...')
    urllib.request.urlretrieve('https://github.com/karolpiczak/ESC-50/archive/master.zip', 'esc50.zip')
    with zipfile.ZipFile('esc50.zip', 'r') as zf:
        zf.extractall('.')
    print('ESC-50 dataset extracted successfully.')

In [ ]:
# Organizing ESC-50 Dog (cat 0), Traffic/Car Horn/Engine/Siren (cats 40,41,42,43), and Background into dataset/ folders
import csv, shutil
for c in ['dog', 'dj', 'traffic', 'background']:
    os.makedirs(f'dataset/{c}', exist_ok=True)

meta_csv = 'ESC-50-master/meta/esc50.csv'
if os.path.exists(meta_csv):
    with open(meta_csv, 'r') as f:
        for row in csv.DictReader(f):
            cat = row['category']
            src = f"ESC-50-master/audio/{row['filename']}"
            if cat == 'dog':
                shutil.copy(src, f"dataset/dog/{row['filename']}")
            elif cat in ('car_horn', 'engine', 'siren', 'helicopter'):
                shutil.copy(src, f"dataset/traffic/{row['filename']}")
            elif cat in ('clock_tick', 'wind', 'rain', 'breathing', 'keyboard_typing'):
                shutil.copy(src, f"dataset/background/{row['filename']}")
    print('ESC-50 clips partitioned into dataset directories.')

In [ ]:
# Execute knowledge distillation and INT8 quantization script:
if os.path.exists('distill_from_yamnet.py'):
    !python3 distill_from_yamnet.py --data-dir dataset --epochs 35
else:
    raise FileNotFoundError('Required pipeline scripts (distill_from_yamnet.py) not found in execution directory.')